# Connected boiler steam-path initialization and numerical scaling

```text
ECON → Water_wall → PrSH → PlSH → FSH → ATMP1
```

All units and Arcs are created first. Units are then initialized and scaled sequentially from upstream to downstream. A final non-overwriting AutoScaler pass covers the assembled flowsheet and connection equations.


In [1]:
import logging
import numpy as np
from scipy.sparse.linalg import norm as sparse_norm
import pyomo.environ as pyo
from pyomo.environ import value, units as pyunits
from pyomo.network import Arc
from pyomo.opt import check_optimal_termination

from idaes.core import FlowsheetBlock
from idaes.core.solvers import get_solver
from idaes.core.util.initialization import propagate_state as _set_port
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util import DiagnosticsToolbox
from idaes.core.scaling import AutoScaler
from idaes.core.scaling.util import get_jacobian, jacobian_cond, list_unscaled_constraints, list_unscaled_variables
from idaes.models.properties import iapws95
from idaes.models.unit_models import Heater, Mixer
from idaes.models.unit_models.separator import (
    Separator, SplittingType, EnergySplittingType,
)
from idaes.models_extra.power_generation.properties import FlueGasParameterBlock
from idaes.models_extra.power_generation.unit_models.boiler_heat_exchanger import BoilerHeatExchanger, HeatExchangerFlowPattern, TubeArrangement

## Build all units and connections

In [2]:
m = pyo.ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.prop_water = iapws95.Iapws95ParameterBlock()
m.fs.prop_fluegas = FlueGasParameterBlock()

def boiler_hx(phase, radiation):
    return BoilerHeatExchanger(
        cold_side={"property_package": m.fs.prop_water, "has_pressure_change": True},
        hot_side={"property_package": m.fs.prop_fluegas, "has_pressure_change": True},
        has_holdup=False,
        flow_pattern=HeatExchangerFlowPattern.countercurrent,
        tube_arrangement=TubeArrangement.inLine,
        cold_side_water_phase=phase,
        has_radiation=radiation,
    )

m.fs.ECON = boiler_hx("Liq", False)
m.fs.Water_wall = Heater(property_package=m.fs.prop_water)
m.fs.PrSH = boiler_hx("Vap", True)
m.fs.PlSH = Heater(property_package=m.fs.prop_water)
m.fs.FSH = boiler_hx("Vap", True)
m.fs.RH = boiler_hx("Vap", True)
m.fs.Spl1 = Separator(
    property_package=m.fs.prop_fluegas,
    split_basis=SplittingType.totalFlow,
    energy_split_basis=EnergySplittingType.equal_temperature,
)
m.fs.mix1 = Mixer(
    property_package=m.fs.prop_fluegas,
    inlet_list=["Reheat_out", "PrSH_out"],
    dynamic=False,
)
m.fs.ATMP1 = Mixer(property_package=m.fs.prop_water, inlet_list=["Steam", "SprayWater"], dynamic=False)

m.fs.econ_to_ww = Arc(source=m.fs.ECON.cold_side_outlet, destination=m.fs.Water_wall.inlet)
m.fs.ww_to_prsh = Arc(source=m.fs.Water_wall.outlet, destination=m.fs.PrSH.cold_side_inlet)
m.fs.prsh_to_plsh = Arc(source=m.fs.PrSH.cold_side_outlet, destination=m.fs.PlSH.inlet)
m.fs.plsh_to_fsh = Arc(source=m.fs.PlSH.outlet, destination=m.fs.FSH.cold_side_inlet)
m.fs.fsh_to_atmp1 = Arc(source=m.fs.FSH.cold_side_outlet, destination=m.fs.ATMP1.Steam)

# Connected flue-gas route from the original boiler subflowsheet.
m.fs.fg_fsh_to_split = Arc(source=m.fs.FSH.hot_side_outlet, destination=m.fs.Spl1.inlet)
m.fs.fg_split_to_rh = Arc(source=m.fs.Spl1.outlet_1, destination=m.fs.RH.hot_side_inlet)
m.fs.fg_split_to_prsh = Arc(source=m.fs.Spl1.outlet_2, destination=m.fs.PrSH.hot_side_inlet)
m.fs.fg_rh_to_mix = Arc(source=m.fs.RH.hot_side_outlet, destination=m.fs.mix1.Reheat_out)
m.fs.fg_prsh_to_mix = Arc(source=m.fs.PrSH.hot_side_outlet, destination=m.fs.mix1.PrSH_out)
m.fs.fg_mix_to_econ = Arc(source=m.fs.mix1.outlet, destination=m.fs.ECON.hot_side_inlet)
pyo.TransformationFactory("network.expand_arcs").apply_to(m.fs)

initialization_options = {"tol":1e-8, "max_iter":100, "linear_solver":"ma27", "nlp_scaling_method":"gradient-based"}


## Shared scaling helper

In [3]:
def autoscale_unit(unit, label):
    scaler = AutoScaler(overwrite=False, zero_tolerance=1e-12,
        min_variable_scaling_factor=1e-10, max_variable_scaling_factor=1e10,
        min_constraint_scaling_factor=1e-10, max_constraint_scaling_factor=1e10)
    scaler.scale_variables_by_magnitude(unit, descend_into=True)
    scaler.scale_constraints_by_jacobian_norm(unit, norm=2, descend_into=True)
    uv = list_unscaled_variables(unit, descend_into=True, include_fixed=False)
    uc = list_unscaled_constraints(unit, descend_into=True)
    print(f"{label}: unscaled active variables={len(uv)}, constraints={len(uc)}")
    return scaler


## 1. Economizer

In [4]:
u=m.fs.ECON
u.cold_side_inlet.flow_mol[0].fix(24194.177); u.cold_side_inlet.enth_mol[0].fix(14990.97); u.cold_side_inlet.pressure[0].fix(26922222.222)
fg=21290.6999; comp={"H2O":.0869,"CO2":.1449,"N2":.7434,"O2":.0247,"NO":.0006,"SO2":.002}
for j,x in comp.items(): u.hot_side_inlet.flow_mol_comp[0,j].fix(fg*x)
u.hot_side_inlet.temperature[0].fix(682.335); u.hot_side_inlet.pressure[0].fix(100145)
I=.0254
u.tube_di.fix((2-2*.188)*I); u.tube_thickness.fix(.188*I); u.pitch_x.fix(3.5*I); u.pitch_y.fix(5.03*I)
u.tube_length.fix(53.41*12*I); u.tube_nrow.fix(36*2.5); u.tube_ncol.fix(130); u.nrow_inlet.fix(2); u.delta_elevation.fix(50)
u.tube_r_fouling=.000176; u.shell_r_fouling=.00088
u.fcorrection_htc.fix(1.5); u.fcorrection_dp_tube.fix(1); u.fcorrection_dp_shell.fix(1)
print("ECON DOF:",degrees_of_freedom(u)); u.initialize(outlvl=logging.INFO,optarg=initialization_options); econ_scaler=autoscale_unit(u,"ECON")


ECON DOF: 0
2026-09-29 13:31:08 [INFO] idaes.init.fs.ECON.cold_side: Initialization Complete


2026-09-29 13:31:08 [INFO] idaes.init.fs.ECON.hot_side.properties_in: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON.hot_side.properties_out: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON.hot_side.properties_out: fs.ECON.hot_side.properties_out State Released.


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON.hot_side: Initialization Complete


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON: fs.ECON Initialisation Step 1 Complete.


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON.hot_side.properties_in: fs.ECON.hot_side.properties_in State Released.


2026-09-29 13:31:09 [INFO] idaes.init.fs.ECON: fs.ECON Initialisation Complete.


component keys that are not exported as part of the NL file.  Skipping.


ECON: unscaled active variables=0, constraints=0


## 2. Water wall

In [5]:
_set_port(arc=m.fs.econ_to_ww)
u=m.fs.Water_wall; u.heat_duty[0].fix(7.51e8)
u.initialize(outlvl=logging.INFO,optarg=initialization_options); ww_scaler=autoscale_unit(u,"Water_wall")
print("Water_wall inlet/outlet [K]:",value(u.control_volume.properties_in[0].temperature),value(u.control_volume.properties_out[0].temperature))


2026-09-29 13:31:10 [INFO] idaes.init.fs.Water_wall.control_volume: Initialization Complete


2026-09-29 13:31:10 [INFO] idaes.init.fs.Water_wall: Initialization Complete: optimal - Optimal Solution Found


component keys that are not exported as part of the NL file.  Skipping.


Water_wall: unscaled active variables=0, constraints=0
Water_wall inlet/outlet [K]: 533.8799852225646 713.057945689533


## 3. Primary superheater

In [6]:
u=m.fs.PrSH
# Only the external hot-side boundary is fixed; the cold inlet comes from Water_wall.
fg=21290.6999*.18
for j,x in comp.items(): u.hot_side_inlet.flow_mol_comp[0,j].fix(fg*x)
u.hot_side_inlet.temperature[0].fix(1180.335); u.hot_side_inlet.pressure[0].fix(100145)
u.tube_di.fix((2.5-2*.165)*I); u.tube_thickness.fix(.165*I); u.pitch_x.fix(3*I); u.pitch_y.fix(54.78/108*12*I)
u.tube_length.fix(53.13*12*I); u.tube_nrow.fix(40); u.tube_ncol.fix(108); u.nrow_inlet.fix(4); u.delta_elevation.fix(50)
u.tube_r_fouling=.000176; u.shell_r_fouling=.003131; u.emissivity_wall.fix(.7)
u.fcorrection_htc.fix(1.5); u.fcorrection_dp_tube.fix(1); u.fcorrection_dp_shell.fix(1)
_set_port(arc=m.fs.ww_to_prsh)
u.initialize(outlvl=logging.INFO,optarg=initialization_options); prsh_scaler=autoscale_unit(u,"PrSH")


2026-09-29 13:31:10 [INFO] idaes.init.fs.PrSH.cold_side: Initialization Complete


2026-09-29 13:31:10 [INFO] idaes.init.fs.PrSH.hot_side.properties_in: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH.hot_side.properties_out: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH.hot_side.properties_out: fs.PrSH.hot_side.properties_out State Released.


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH.hot_side: Initialization Complete


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH: fs.PrSH Initialisation Step 1 Complete.


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH.hot_side.properties_in: fs.PrSH.hot_side.properties_in State Released.


2026-09-29 13:31:11 [INFO] idaes.init.fs.PrSH: fs.PrSH Initialisation Complete.


component keys that are not exported as part of the NL file.  Skipping.


PrSH: unscaled active variables=0, constraints=0


## 4. Platen superheater

In [7]:
_set_port(arc=m.fs.prsh_to_plsh)
u=m.fs.PlSH; u.heat_duty[0].fix(5.5e7)
u.initialize(outlvl=logging.INFO,optarg=initialization_options); plsh_scaler=autoscale_unit(u,"PlSH")


2026-09-29 13:31:11 [INFO] idaes.init.fs.PlSH.control_volume: Initialization Complete


2026-09-29 13:31:12 [INFO] idaes.init.fs.PlSH: Initialization Complete: optimal - Optimal Solution Found


component keys that are not exported as part of the NL file.  Skipping.


PlSH: unscaled active variables=0, constraints=0


## 5. Finishing superheater

In [8]:
u=m.fs.FSH
fg=21290.6999
for j,x in comp.items(): u.hot_side_inlet.flow_mol_comp[0,j].fix(fg*x)
u.hot_side_inlet.temperature[0].fix(1300.335); u.hot_side_inlet.pressure[0].fix(100145)
u.tube_di.fix((2.5-2*.165)*I); u.tube_thickness.fix(.165*I); u.pitch_x.fix(3*I); u.pitch_y.fix(54.78/108*12*I)
u.tube_length.fix(53.13*12*I); u.tube_nrow.fix(16); u.tube_ncol.fix(85); u.nrow_inlet.fix(2); u.delta_elevation.fix(50)
u.tube_r_fouling=.000176; u.shell_r_fouling=.003131; u.emissivity_wall.fix(.7)
u.fcorrection_htc.fix(1); u.fcorrection_dp_tube.fix(1); u.fcorrection_dp_shell.fix(1)
_set_port(arc=m.fs.plsh_to_fsh)
u.initialize(outlvl=logging.INFO,optarg=initialization_options); fsh_scaler=autoscale_unit(u,"FSH")


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH.cold_side: Initialization Complete


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH.hot_side.properties_in: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH.hot_side.properties_out: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH.hot_side.properties_out: fs.FSH.hot_side.properties_out State Released.


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH.hot_side: Initialization Complete


2026-09-29 13:31:12 [INFO] idaes.init.fs.FSH: fs.FSH Initialisation Step 1 Complete.


2026-09-29 13:31:13 [INFO] idaes.init.fs.FSH.hot_side.properties_in: fs.FSH.hot_side.properties_in State Released.


2026-09-29 13:31:13 [INFO] idaes.init.fs.FSH: fs.FSH Initialisation Complete.


component keys that are not exported as part of the NL file.  Skipping.


FSH: unscaled active variables=0, constraints=0


## 6. Main-steam attemperator

In [9]:
# Reheater: separate HP-turbine-exhaust steam circuit.
u = m.fs.RH
u.cold_side_inlet.flow_mol[0].fix(21235.27)
u.cold_side_inlet.enth_mol[0].fix(53942.7569)
u.cold_side_inlet.pressure[0].fix(3677172.33638)

# Temporary hot-side values are used only for unit initialization. They are
# released below so the splitter arc determines the final inlet state.
rh_fg = 21290.6999 * 0.75
for j, x in comp.items():
    u.hot_side_inlet.flow_mol_comp[0, j].fix(rh_fg * x)
u.hot_side_inlet.temperature[0].fix(1180.335)
u.hot_side_inlet.pressure[0].fix(100145)

u.tube_di.fix((2.5 - 2 * 0.165) * I)
u.tube_thickness.fix(0.11 * I)
u.pitch_x.fix(3 * I)
u.pitch_y.fix(54.08 / 108 * 12 * I)
u.tube_length.fix(53.82 * 12 * I)
u.tube_nrow.fix(36)
u.tube_ncol.fix(152)
u.nrow_inlet.fix(2)
u.delta_elevation.fix(30)
u.tube_r_fouling = 0.000176
u.shell_r_fouling = 0.00088
u.emissivity_wall.fix(0.7)
u.fcorrection_htc.fix(1.8)
u.fcorrection_dp_tube.fix(1)
u.fcorrection_dp_shell.fix(1)
print("RH DOF:", degrees_of_freedom(u))
u.initialize(outlvl=logging.INFO, optarg=initialization_options)
rh_scaler = autoscale_unit(u, "RH")

RH DOF: 0
2026-09-29 13:31:13 [INFO] idaes.init.fs.RH.cold_side: Initialization Complete


2026-09-29 13:31:13 [INFO] idaes.init.fs.RH.hot_side.properties_in: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH.hot_side.properties_out: Initialisation Complete, optimal - Optimal Solution Found.


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH.hot_side.properties_out: fs.RH.hot_side.properties_out State Released.


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH.hot_side: Initialization Complete


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH: fs.RH Initialisation Step 1 Complete.


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH.hot_side.properties_in: fs.RH.hot_side.properties_in State Released.


2026-09-29 13:31:14 [INFO] idaes.init.fs.RH: fs.RH Initialisation Complete.


component keys that are not exported as part of the NL file.  Skipping.


RH: unscaled active variables=0, constraints=0


In [10]:
# Initialize the splitter and mixer with temporary independent inlet states.
splitter = m.fs.Spl1
splitter.split_fraction[0, "outlet_1"].fix(0.75)
for j in comp:
    splitter.inlet.flow_mol_comp[0, j].fix(
        value(m.fs.FSH.hot_side_outlet.flow_mol_comp[0, j])
    )
splitter.inlet.temperature[0].fix(value(m.fs.FSH.hot_side_outlet.temperature[0]))
splitter.inlet.pressure[0].fix(value(m.fs.FSH.hot_side_outlet.pressure[0]))
splitter.initialize(outlvl=logging.INFO, optarg=initialization_options)
splitter_scaler = autoscale_unit(splitter, "Spl1")

mix = m.fs.mix1
for inlet, outlet in (
    (mix.Reheat_out, m.fs.RH.hot_side_outlet),
    (mix.PrSH_out, m.fs.PrSH.hot_side_outlet),
):
    for j in comp:
        inlet.flow_mol_comp[0, j].fix(value(outlet.flow_mol_comp[0, j]))
    inlet.temperature[0].fix(value(outlet.temperature[0]))
    inlet.pressure[0].fix(value(outlet.pressure[0]))
mix.initialize(outlvl=logging.INFO, optarg=initialization_options)
mix_scaler = autoscale_unit(mix, "mix1")

# Release only states determined by the connected flue-gas arcs. FSH hot-side
# inlet remains the external boiler-gas boundary; RH cold-side inlet remains
# the external HP-turbine-exhaust boundary.
connected_inlet_ports = (
    m.fs.Spl1.inlet,
    m.fs.RH.hot_side_inlet,
    m.fs.PrSH.hot_side_inlet,
    m.fs.mix1.Reheat_out,
    m.fs.mix1.PrSH_out,
    m.fs.ECON.hot_side_inlet,
)
for port in connected_inlet_ports:
    for variable in port.vars.values():
        variable.unfix()

# Populate consistent starting values along the gas path.
_set_port(arc=m.fs.fg_fsh_to_split)
_set_port(arc=m.fs.fg_split_to_rh)
_set_port(arc=m.fs.fg_split_to_prsh)
_set_port(arc=m.fs.fg_rh_to_mix)
_set_port(arc=m.fs.fg_prsh_to_mix)
_set_port(arc=m.fs.fg_mix_to_econ)
print("Connected flue-gas path: FSH -> Spl1 -> (RH, PrSH) -> mix1 -> ECON")

2026-09-29 13:31:14 [INFO] idaes.init.fs.Spl1.mixed_state: Initialisation Complete, skipped.


2026-09-29 13:31:14 [INFO] idaes.init.fs.Spl1.outlet_1_state: Initialisation Complete, skipped.


2026-09-29 13:31:14 [INFO] idaes.init.fs.Spl1.outlet_1_state: fs.Spl1.outlet_1_state State Released.


2026-09-29 13:31:14 [INFO] idaes.init.fs.Spl1.outlet_2_state: Initialisation Complete, skipped.


2026-09-29 13:31:14 [INFO] idaes.init.fs.Spl1.outlet_2_state: fs.Spl1.outlet_2_state State Released.


...

2026-09-29 13:31:15 [INFO] idaes.init.fs.mix1: Initialization Complete: optimal - Optimal Solution Found


2026-09-29 13:31:15 [INFO] idaes.init.fs.mix1.Reheat_out_state: fs.mix1.Reheat_out_state State Released.


2026-09-29 13:31:15 [INFO] idaes.init.fs.mix1.PrSH_out_state: fs.mix1.PrSH_out_state State Released.


component keys that are not exported as part of the NL file.  Skipping.


mix1: unscaled active variables=0, constraints=0
Connected flue-gas path: FSH -> Spl1 -> (RH, PrSH) -> mix1 -> ECON


In [11]:
u=m.fs.ATMP1
hs=value(iapws95.htpx(563.15*pyunits.K,2.5449e7*pyunits.Pa))
u.SprayWater_state[0].flow_mol.fix(.001); u.SprayWater_state[0].pressure.fix(1.22e8); u.SprayWater_state[0].enth_mol.fix(hs)
_set_port(arc=m.fs.fsh_to_atmp1)
u.initialize(outlvl=logging.INFO,optarg=initialization_options); atmp_scaler=autoscale_unit(u,"ATMP1")


2026-09-29 13:31:16 [INFO] idaes.init.fs.ATMP1: Initialization Complete: optimal - Optimal Solution Found


component keys that are not exported as part of the NL file.  Skipping.


ATMP1: unscaled active variables=0, constraints=0


## Scale and solve the complete connected steam path

In [12]:
print("Full flowsheet DOF:",degrees_of_freedom(m.fs))
if degrees_of_freedom(m.fs)!=0: raise RuntimeError("Connected flowsheet is not square")
fs_scaler=AutoScaler(overwrite=False)
fs_scaler.scale_variables_by_magnitude(m.fs,descend_into=True)
fs_scaler.scale_constraints_by_jacobian_norm(m.fs,norm=2,descend_into=True)
print("Unscaled active variables:",len(list_unscaled_variables(m.fs,descend_into=True,include_fixed=False)))
print("Unscaled constraints:",len(list_unscaled_constraints(m.fs,descend_into=True)))
solver=get_solver(); solver.options={"tol":1e-8,"max_iter":100,"linear_solver":"ma27","halt_on_ampl_error":"yes","nlp_scaling_method":"user-scaling"}
results=solver.solve(m.fs,tee=True,symbolic_solver_labels=True)
print(results.solver.status,results.solver.termination_condition)
if not check_optimal_termination(results): raise RuntimeError("Connected solve failed")


Full flowsheet DOF: 0
component keys that are not exported as part of the NL file.  Skipping.


Unscaled active variables: 0
Unscaled constraints: 0
component keys that are not exported as part of the NL file.  Skipping.


Ipopt 3.13.2: tol=1e-08
max_iter=100
linear_solver=ma27
halt_on_ampl_error=yes
nlp_scaling_method=user-scaling




******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://www.hsl.rl.ac.

   4  0.0000000e+00 3.41e-06 1.00e-06  -1.0 1.66e-06    -  1.00e+00 1.00e+00h  1

Number of Iterations....: 4

                                   (scaled)                 (unscaled)
Objective...............:   0.0000000000000000e+00    0.0000000000000000e+00
Dual infeasibility......:   0.0000000000000000e+00    0.0000000000000000e+00
Constraint violation....:   2.8082958075017172e-14    3.4123659133911133e-06
Complementarity.........:   0.0000000000000000e+00    0.0000000000000000e+00
Overall NLP error.......:   2.8082958075017172e-14    3.4123659133911133e-06


Number of objective function evaluations             = 5
Number of objective gradient evaluations             = 5
Number of equality constraint evaluations            = 5
Number of inequality constraint evaluations          = 0
Number of equality constraint Jacobian evaluations   = 5
Number of inequality constraint Jacobian evaluations = 0
Number of Lagrangian Hessian evaluations             = 4
Total CPU secs in IPOPT (w/o fun

ok optimal


## Full-model Jacobian diagnostics

In [13]:
# Sparse Jacobian diagnostics. Dense condition-number inversion is omitted
# because it terminates the kernel for this full radiative flowsheet.
js, nlp = get_jacobian(
    m.fs,
    include_scaling_factors=True,
    equality_constraints_only=True,
)
rn = np.asarray(abs(js).sum(axis=1)).ravel()
cn = np.asarray(abs(js).sum(axis=0)).ravel()
constraints = nlp.get_pyomo_constraints()
variables = nlp.get_pyomo_variables()

print(f"Scaled row norms: {rn.min():.3e} to {rn.max():.3e}")
print(f"Scaled column norms: {cn.min():.3e} to {cn.max():.3e}")

print("\nExtreme Jacobian rows:")
for i, norm in enumerate(rn):
    if norm < 1e-4 or norm > 1e4:
        print(f"  {constraints[i].name}: {norm:.3e}")

print("\nExtreme Jacobian columns:")
for j, norm in enumerate(cn):
    if norm < 1e-4 or norm > 1e4:
        print(f"  {variables[j].name}: {norm:.3e}")

component keys that are not exported as part of the NL file.  Skipping.


Scaled row norms: 1.000e+00 to 5.094e+00
Scaled column norms: 9.854e-02 to 1.312e+01

Extreme Jacobian rows:

Extreme Jacobian columns:


## Connected process results

In [14]:
print("ECON outlet [K]:",value(m.fs.ECON.cold_side.properties_out[0].temperature))
print("Water_wall outlet [K]:",value(m.fs.Water_wall.control_volume.properties_out[0].temperature))
print("PrSH outlet [K]:",value(m.fs.PrSH.cold_side.properties_out[0].temperature))
print("PlSH outlet [K]:",value(m.fs.PlSH.control_volume.properties_out[0].temperature))
print("FSH outlet [K]:",value(m.fs.FSH.cold_side.properties_out[0].temperature))
print("ATMP1 outlet [K]:",value(m.fs.ATMP1.mixed_state[0].temperature))

print("RH steam outlet [K]:", value(m.fs.RH.cold_side.properties_out[0].temperature))
print("RH gas outlet [K]:", value(m.fs.RH.hot_side.properties_out[0].temperature))
print("Flue-gas split to RH:", value(m.fs.Spl1.split_fraction[0, "outlet_1"]))
print("ECON gas inlet [K]:", value(m.fs.ECON.hot_side.properties_in[0].temperature))

ECON outlet [K]: 546.8808328605156
Water_wall outlet [K]: 723.9095101569857
PrSH outlet [K]: 764.1677527834819
PlSH outlet [K]: 797.4895466622572
FSH outlet [K]: 873.3699075510588
ATMP1 outlet [K]: 873.3698772408015
RH steam outlet [K]: 871.6876469877258
RH gas outlet [K]: 716.1530530112951
Flue-gas split to RH: 0.75
ECON gas inlet [K]: 723.2885047410446
